In [1]:
# 直接拆解数据，使用最小二乘法对模型进行参数辨识
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import math, datetime


In [2]:
data = pd.read_csv("../../log/realdata/StaticProcess/real_StaticPoint_6group_2025-02-21_08-55-07.csv")
# 实验数据
P1_array = data['P1'].values
P2_array = data['P2'].values
theta1_array = data['theta1'].values
theta2_array = data['theta2'].values
# 实验数据
theta1 = torch.tensor(theta1_array, dtype=torch.float32)*torch.pi/180  # 输入 theta1
theta2 = torch.tensor(theta2_array, dtype=torch.float32)*torch.pi/180  # 输入 theta2
P1_actual = torch.tensor(P1_array, dtype=torch.float32)*1000  # 实验输出 P
P2_actual = torch.tensor(P2_array, dtype=torch.float32)*1000  # 实验输出 P
P2_actual.shape

torch.Size([13])

In [3]:
# make the data from the geometry model
# 为了简单, make data 不用 math
def get_geom_data(theta_1, theta_2):
    '''
        return: gLinvY, l1, l2 (各个分量)
    '''
    # 固定参数
    a_1, a_2, b_1, b_2, d_1, d_2 = 0.25, 0.25, 0.21213, 0.1, 0.06, 0.10
    beta_1, beta_2 = 8.13 / 180 * math.pi, 30 / 180 * math.pi
    g = 9.8

    A_x_O = d_1
    A_y_O = 0

    B_x_O = -d_2
    B_y_O = 0

    C_x_O = b_1 * math.cos(theta_1 - beta_1)
    C_y_O = b_1 * math.sin(theta_1 - beta_1)

    D_x_O = a_1 * math.cos(theta_1) + b_2 * math.cos(theta_1 + theta_2 + beta_2)
    D_y_O = a_1 * math.sin(theta_1) + b_2 * math.sin(theta_1 + theta_2 + beta_2)

    E_x_O = a_1 * math.cos(theta_1)
    E_y_O = a_1 * math.sin(theta_1)

    F_x_O = a_1 * math.cos(theta_1) + a_2 * math.cos(theta_1 + theta_2)
    F_y_O = a_1 * math.sin(theta_1) + a_2 * math.sin(theta_1 + theta_2)

    # 计算偏导数
    # 对 theta_1 的偏导数

    dA_x_O_dtheta_1 = 0
    dA_y_O_dtheta_1 = 0

    dB_x_O_dtheta_1 = 0
    dB_y_O_dtheta_1 = 0

    dC_x_O_dtheta_1 = -b_1 * math.sin(theta_1 - beta_1)
    dC_y_O_dtheta_1 = b_1 * math.cos(theta_1 - beta_1)

    dD_x_O_dtheta_1 = -a_1 * math.sin(theta_1) - b_2 * math.sin(theta_1 + theta_2 + beta_2)
    dD_y_O_dtheta_1 = a_1 * math.cos(theta_1) + b_2 * math.cos(theta_1 + theta_2 + beta_2)

    dE_x_O_dtheta_1 = -a_1 * math.sin(theta_1)
    dE_y_O_dtheta_1 = a_1 * math.cos(theta_1)

    dF_x_O_dtheta_1 = -a_1 * math.sin(theta_1) - a_2 * math.sin(theta_1 + theta_2)
    dF_y_O_dtheta_1 = a_1 * math.cos(theta_1) + a_2 * math.cos(theta_1 + theta_2)

    # 对 theta_2 的偏导数

    dA_x_O_dtheta_2 = 0.0
    dA_y_O_dtheta_2 = 0.0

    dB_x_O_dtheta_2 = 0.0
    dB_y_O_dtheta_2 = 0.0

    dC_x_O_dtheta_2 = 0.0
    dC_y_O_dtheta_2 = 0.0

    dD_x_O_dtheta_2 = -b_2 * math.sin(theta_1 + theta_2 + beta_2)
    dD_y_O_dtheta_2 = b_2 * math.cos(theta_1 + theta_2 + beta_2)

    dE_x_O_dtheta_2 = 0.0
    dE_y_O_dtheta_2 = 0.0

    dF_x_O_dtheta_2 = -a_2 * math.sin(theta_1 + theta_2)
    dF_y_O_dtheta_2 = a_2 * math.cos(theta_1 + theta_2)

    # 计算长度 l1 和 l2
    l_1 = math.sqrt((A_x_O - C_x_O)**2 + (A_y_O - C_y_O)**2)
    l_2 = math.sqrt((B_x_O - D_x_O)**2 + (B_y_O - D_y_O)**2)

    # 计算偏导数
    # 偏导数 d/dtheta_1
    dl_1_dtheta_1 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_1 - dC_x_O_dtheta_1) + (A_y_O - C_y_O) * (dA_y_O_dtheta_1 - dC_y_O_dtheta_1))
    dl_2_dtheta_1 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_1 - dD_x_O_dtheta_1) + (B_y_O - D_y_O) * (dB_y_O_dtheta_1 - dD_y_O_dtheta_1))

    # 偏导数 d/dtheta_2
    dl_1_dtheta_2 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_2 - dC_x_O_dtheta_2) + (A_y_O - C_y_O) * (dA_y_O_dtheta_2 - dC_y_O_dtheta_2))
    dl_2_dtheta_2 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_2 - dD_x_O_dtheta_2) + (B_y_O - D_y_O) * (dB_y_O_dtheta_2 - dD_y_O_dtheta_2))

    # print(dl_1_dtheta_2)  # check the model

    # 等式右侧
    # print(type(dE_y_O_dtheta_2/2))
    # print(type((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2))
    # print(type(dC_y_O_dtheta_2/2))
    # print(type(dD_y_O_dtheta_2/2))
    
    # _ = math.stack([(dE_y_O_dtheta_2/2), ((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2), (dC_y_O_dtheta_2/2), (dD_y_O_dtheta_2/2)], dim=1)
    Y_mat = np.array([
        [(dE_y_O_dtheta_1/2), ((dE_y_O_dtheta_1+dF_y_O_dtheta_1)/2), (dC_y_O_dtheta_1/2), (dD_y_O_dtheta_1/2)], 
        [(dE_y_O_dtheta_2/2), ((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2), (dC_y_O_dtheta_2/2), (dD_y_O_dtheta_2/2)]])

    # 等式左侧
    L_mat = np.array([
        [dl_1_dtheta_1, dl_2_dtheta_1],
        [dl_1_dtheta_2, dl_2_dtheta_2]])

    return Y_mat, L_mat, l_1, l_2



In [9]:
# 构造最小二乘问题（补变量Trick）——气压模型
# 变量集：S1, S2, m1, m2, m3, m4, k1, k2, -k1*l10, -k2*l20
var_num = 4
data_size = len(theta1)
A_mat = np.zeros((data_size*2, var_num))
b_mat = np.zeros((data_size*2, 1))
m1, m2, m3, m4 = 0.086, 0.1033, 186.48 * 1e-3, 272.66 * 1e-3
g = 9.8
l10_set, l20_set = 0.164, 0.258
m_array = np.array([m1, m2, m3, m4]).reshape(-1, 1)

for i in range(data_size):      # 第 i 大组
    t1, t2 = theta1[i], theta2[i]
    Y_mat, L_mat, l_1, l_2 = get_geom_data(t1, t2)
    # print(Y_mat, L_mat, l_1, l_2)
    temp_mat = -g*np.linalg.inv(L_mat)@Y_mat@m_array
    A_mat[i*2, 0] = temp_mat[0, 0]
    A_mat[i*2, 1] = 0
    # A_mat[i*2, 2] = temp_mat[0, 0]
    # A_mat[i*2, 3] = temp_mat[0, 1]
    # A_mat[i*2, 4] = temp_mat[0, 2]
    # A_mat[i*2, 5] = temp_mat[0, 3]
    A_mat[i*2, 2] = l_1 - l10_set
    A_mat[i*2, 3] = 0
    # A_mat[i*2, 4] = 1
    # A_mat[i*2, 5] = 0

    A_mat[i*2+1, 0] = 0
    A_mat[i*2+1, 1] = temp_mat[1, 0]
    # A_mat[i*2+1, 2] = temp_mat[1, 0]
    # A_mat[i*2+1, 3] = temp_mat[1, 1]
    # A_mat[i*2+1, 4] = temp_mat[1, 2]
    # A_mat[i*2+1, 5] = temp_mat[1, 3]
    A_mat[i*2+1, 2] = 0
    A_mat[i*2+1, 3] = l_2 - l20_set
    # A_mat[i*2+1, 4] = 0
    # A_mat[i*2+1, 5] = 1

    b_mat[i*2, 0] = P1_actual[i]
    b_mat[i*2+1, 0] = P2_actual[i]
    

# sove Ax = 0
solution = np.linalg.inv(A_mat.T@A_mat)@A_mat.T@b_mat
s1_prime = 1/solution[0, 0]
s2_prime = 1/solution[1, 0]
k1_prime = solution[2, 0]*s1_prime
k2_prime = solution[3, 0]*s2_prime
# l10_prime = -solution[4, 0]/k1_prime*s1_prime
# l20_prime = -solution[5, 0]/k2_prime*s2_prime
l10_prime = l10_set
l20_prime = l20_set

# print the result
print(f"s1: {s1_prime}, s2: {s2_prime}, \nk1: {k1_prime}, k2: {k2_prime}, \nl10: {l10_prime}, l20: {l20_prime}")

x_mat = A_mat@solution
x_mat = x_mat.reshape(-1, 2)
p_mat = np.stack((P1_actual, P2_actual), axis=1)

# print(x_mat)
# print(p_mat)
print(f"PressureError(sqrtMSE): {math.sqrt(np.linalg.norm(x_mat-p_mat)**2/2/data_size)/1000:.6f}")

# 力大小
F_mat = p_mat
F_mat[:, 0] = F_mat[:, 0]*s1_prime
F_mat[:, 1] = F_mat[:, 1]*s2_prime
print(F_mat)
print(theta1[2]-(-0.295+math.pi/2), (theta2[2]-0.569))

theta_mat = np.stack([theta1, theta2], axis=1)
theta_mat

s1: 0.0003538647203395965, s2: 0.00034663180121508557, 
k1: 141.0296084110311, k2: 113.79784850542737, 
l10: 0.164, l20: 0.258
PressureError(sqrtMSE): 0.842839
[[ 0.         0.       ]
 [ 0.         3.4663181]
 [ 0.         6.9326363]
 [ 0.        10.398954 ]
 [ 3.5386472  0.       ]
 [ 3.5386472  3.4663181]
 [ 3.5386472  6.9326363]
 [ 7.0772943  0.       ]
 [ 7.0772943  3.4663181]
 [10.615942   0.       ]
 [10.615942   3.4663181]
 [14.154589   0.       ]
 [17.693235   0.       ]]
tensor(0.0071) tensor(-0.0530)


array([[1.2755772 , 1.0714235 ],
       [1.3015182 , 0.7927732 ],
       [1.2829165 , 0.51602167],
       [1.2929788 , 0.21563765],
       [1.4700553 , 0.8628211 ],
       [1.4716564 , 0.5444165 ],
       [1.4977674 , 0.199207  ],
       [1.6180997 , 0.69558835],
       [1.6336076 , 0.34381413],
       [1.7746446 , 0.5192293 ],
       [1.7917793 , 0.18421595],
       [1.9147946 , 0.37423247],
       [2.0679529 , 0.17999372]], dtype=float32)

In [ ]:
# 对比分析 k delta l vs M
for i in range(data_size):      # 第 i 大组
    t1, t2 = theta1[i], theta2[i]
    Y_mat, L_mat, l_1, l_2 = get_geom_data(t1, t2)
    # print(Y_mat, L_mat, l_1, l_2)
    temp_mat = -g*np.linalg.inv(L_mat)@Y_mat@m_array
    print(f"Mass part: {temp_mat[0], temp_mat[1]}")
    print(f"Elastic part: {k1_prime*(l_1-l10_prime), k2_prime*(l_2-l20_prime)}")

In [ ]:
# load npy
# real vs sim(ideal)
print("Real vs Sim(ideal)")
# tmp = np.load("/Users/flypig/Documents/Coding/MujocoLearn/data/Exp-sim-real constrast-20250219_230253/StaticState_list.npy")        # the ideal model
tmp = np.load("/Users/flypig/Documents/Coding/MujocoLearn/data/Exp-sim-real constrast-20250219_231642/StaticState_list.npy")        # the geom model

theta_sim = tmp[:,3:5]
theta_sim[:, 0] = theta_sim[:, 0] + math.pi/2

# for geom model
theta_sim[:, 1] = theta_sim[:, 1] + math.pi/2

print(theta_sim*180/math.pi)

theta_error = (theta_sim - theta_mat)
print(theta_error)
print(f"Error_relative: {theta_error/theta_sim}")
# data_mat for LLM to form a table
data_mat = np.hstack((P1_actual.reshape(data_size, 1)/1000, P2_actual.reshape(data_size, 1)/1000, (theta_sim)*180/math.pi, (theta_mat)*180/math.pi, theta_error*180/math.pi, theta_error/theta_real))
print(data_mat)

# save
dataShow = pd.DataFrame(data_mat, columns=['Pressure1/kPa', 'Pressure2/kPa', 'Theta1_real/deg', 'Theta2_real/deg', 'Theta1_sim/deg', 'Theta2_sim/deg', 'Theta1_error/deg', 'Theta2_error/deg', 'Theta1_errorRelative', 'Theta2_errorRelative'])
# dataShow.to_csv('../../log/matrix_data.csv', index=False)

# in Rad
print(f"Max error: {np.max(np.abs(theta_error))}")
print(f"SqrtMSE: {math.sqrt(np.sum(theta_error**2)/theta_error.size)}")

In [ ]:
# 构造最小二乘问题（补变量Trick）——压力模型，最小化力的MSE，得到模型之后再计算气压的MSE。
# 变量集：S1, S2, k1, k2, -k1*l10, -k2*l20
var_num = 6
data_size = len(theta1)
A_mat = np.zeros((data_size*2, var_num))
b_mat = np.zeros((data_size*2, 1))
m1, m2, m3, m4 = 0.086, 0.1033, 186.48 * 1e-3, 272.66 * 1e-3
g = 9.8
m_array = np.array([m1, m2, m3, m4]).reshape(-1, 1)

for i in range(data_size):      # 第 i 大组
    t1, t2 = theta1[i], theta2[i]
    Y_mat, L_mat, l_1, l_2 = get_geom_data(t1, t2)
    # print(Y_mat, L_mat, l_1, l_2)
    temp_mat = g*np.linalg.inv(L_mat)@Y_mat@m_array
    A_mat[i*2, 0] = -P1_actual[i]
    A_mat[i*2, 1] = 0
    # A_mat[i*2, 2] = temp_mat[0, 0]
    # A_mat[i*2, 3] = temp_mat[0, 1]
    # A_mat[i*2, 4] = temp_mat[0, 2]
    # A_mat[i*2, 5] = temp_mat[0, 3]
    A_mat[i*2, 2] = l_1
    A_mat[i*2, 3] = 0
    A_mat[i*2, 4] = 1
    A_mat[i*2, 5] = 0

    A_mat[i*2+1, 0] = 0
    A_mat[i*2+1, 1] = -P2_actual[i]
    # A_mat[i*2+1, 2] = temp_mat[1, 0]
    # A_mat[i*2+1, 3] = temp_mat[1, 1]
    # A_mat[i*2+1, 4] = temp_mat[1, 2]
    # A_mat[i*2+1, 5] = temp_mat[1, 3]
    A_mat[i*2+1, 2] = 0
    A_mat[i*2+1, 3] = l_2
    A_mat[i*2+1, 4] = 0
    A_mat[i*2+1, 5] = 1

    b_mat[i*2, 0] = temp_mat[0, 0]
    b_mat[i*2+1, 0] = temp_mat[1, 0]
    
# solve Ax = b
solution = np.linalg.inv(A_mat.T@A_mat)@A_mat.T@b_mat
s1_prime = solution[0, 0]
s2_prime = solution[1, 0]
k1_prime = solution[2, 0]
k2_prime = solution[3, 0]
l10_prime = -solution[4, 0]/k1_prime
l20_prime = -solution[5, 0]/k2_prime

# print the result
print(f"s1: {s1_prime}, s2: {s2_prime}, \nk1: {k1_prime}, k2: {k2_prime}, \nl10: {l10_prime}, l20: {l20_prime}")

x_mat = A_mat@solution
print(x_mat - b_mat)
print(x_mat)
x_mat = x_mat.reshape(-1, 2)
x_mat[:,0] = x_mat[:,0]/s1_prime
x_mat[:,1] = x_mat[:,1]/s2_prime
p_mat = np.stack((P1_actual, P2_actual), axis=1)

print(x_mat)
print(p_mat)
print(math.sqrt(np.linalg.norm(x_mat-p_mat)**2/2/data_size)/1000)